In [ ]:
# Bismillah i am writing my first code for pytorch to understand this library best of luck for me

In [ ]:
!pip install opendatasets --quiet
import opendatasets as od
od.download("https://www.kaggle.com/datasets/mssmartypants/rice-type-classification")

Please provide your Kaggle credentials to download this dataset. Learn more: http://bit.ly/kaggle-creds
Your Kaggle username: muntazirmehdi3131
Your Kaggle Key: ··········
Dataset URL: https://www.kaggle.com/datasets/mssmartypants/rice-type-classification


100%|██████████| 888k/888k [00:00<00:00, 78.2MB/s]

In [ ]:
# Headers that are required for the neural network training
import torch
import torch.nn as nn
from torch.optim import Adam
from torch.utils.data import DataLoader , Dataset
from torchsummary import summary
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np


device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)


cpu


In [ ]:
data_df= pd.read_csv("/content/rice-type-classification/riceClassification.csv")
data_df.head()

,id,Area,MajorAxisLength,MinorAxisLength,Eccentricity,ConvexArea,EquivDiameter,Extent,Perimeter,Roundness,AspectRation,Class
0,1,4537,92.229316,64.012769,0.719916,4677,76.004525,0.657536,273.085,0.764510,1.440796,1
1,2,2872,74.691881,51.400454,0.725553,3015,60.471018,0.713009,208.317,0.831658,1.453137,1
2,3,3048,76.293164,52.043491,0.731211,3132,62.296341,0.759153,210.012,0.868434,1.465950,1
3,4,3073,77.033628,51.928487,0.738639,3157,62.551300,0.783529,210.657,0.870203,1.483456,1
4,5,3693,85.124785,56.374021,0.749282,3802,68.571668,0.769375,230.332,0.874743,1.510000,1


In [ ]:
# now drop the missing values and id col as it not giving any valuable info for our model

data_df.dropna(inplace = True)
data_df.drop(['id'] , axis = 1 ,inplace = True )
print(data_df.shape)
data_df.head()

(18185, 11)


,Area,MajorAxisLength,MinorAxisLength,Eccentricity,ConvexArea,EquivDiameter,Extent,Perimeter,Roundness,AspectRation,Class
0,4537,92.229316,64.012769,0.719916,4677,76.004525,0.657536,273.085,0.764510,1.440796,1
1,2872,74.691881,51.400454,0.725553,3015,60.471018,0.713009,208.317,0.831658,1.453137,1
2,3048,76.293164,52.043491,0.731211,3132,62.296341,0.759153,210.012,0.868434,1.465950,1
3,3073,77.033628,51.928487,0.738639,3157,62.551300,0.783529,210.657,0.870203,1.483456,1
4,3693,85.124785,56.374021,0.749282,3802,68.571668,0.769375,230.332,0.874743,1.510000,1


In [ ]:
# check how many classes are here
print(data_df['Class'].unique())
print(data_df['Class'].value_counts())

[1 0]
Class
1    9985
0    8200
Name: count, dtype: int64


In [ ]:
original_df = data_df.copy()

for col in data_df :
  data_df[col] = data_df[col] / data_df[col].abs().max()

data_df.head()

,Area,MajorAxisLength,MinorAxisLength,Eccentricity,ConvexArea,EquivDiameter,Extent,Perimeter,Roundness,AspectRation,Class
0,0.444368,0.503404,0.775435,0.744658,0.424873,0.666610,0.741661,0.537029,0.844997,0.368316,1.0
1,0.281293,0.407681,0.622653,0.750489,0.273892,0.530370,0.804230,0.409661,0.919215,0.371471,1.0
2,0.298531,0.416421,0.630442,0.756341,0.284520,0.546380,0.856278,0.412994,0.959862,0.374747,1.0
3,0.300979,0.420463,0.629049,0.764024,0.286791,0.548616,0.883772,0.414262,0.961818,0.379222,1.0
4,0.361704,0.464626,0.682901,0.775033,0.345385,0.601418,0.867808,0.452954,0.966836,0.386007,1.0


In [ ]:
#not take the last col in x and take the last col only in y
X = np.array(data_df.iloc[:,:-1])
Y = np.array(data_df.iloc[:,-1])

In [ ]:
x_train , x_test , y_train , y_test = train_test_split(X,Y,test_size=0.3)


x_val , x_test , y_val , y_test = train_test_split(x_test,y_test,test_size=0.5)


In [ ]:
print(x_train.shape)
print(x_test.shape)
print(x_val.shape)

(12729, 10)
(2728, 10)
(2728, 10)


In [ ]:
class dataset(Dataset):
  def __init__ (self,X,Y):
    self.X = torch.tensor(X,dtype=torch.float32).to(device)
    self.Y= torch.tensor(Y,dtype=torch.float32).to(device)

  def __len__(self):
    return len(self.X)

  def __getitem__ (self,index):
    return self.X[index] , self.Y[index]

In [ ]:
training_data = dataset(x_train,y_train)
testing_data = dataset(x_test,y_test)
validation_data = dataset(x_val,y_val)


In [ ]:
train_dataloader = DataLoader(training_data,batch_size=8,shuffle=True)
test_loader = DataLoader(testing_data, batch_size=8 , shuffle= True )
validation_loader = DataLoader(validation_data,batch_size=8 , shuffle =True)

In [ ]:
for x, y in train_dataloader:
  print(x)
  print("===================")
  print(y)
  break


tensor([[0.8815, 0.8830, 0.8701, 0.9268, 0.8338, 0.9389, 0.6297, 0.7728, 0.8094,
         0.5758],
        [0.5500, 0.8248, 0.5827, 0.9806, 0.5257, 0.7416, 0.5970, 0.6542, 0.7048,
         0.8030],
        [0.6038, 0.8361, 0.6401, 0.9709, 0.5789, 0.7771, 0.5368, 0.6802, 0.7157,
         0.7411],
        [0.5185, 0.7619, 0.6004, 0.9670, 0.4932, 0.7201, 0.5786, 0.6206, 0.7383,
         0.7199],
        [0.5568, 0.8489, 0.5746, 0.9851, 0.5348, 0.7462, 0.5419, 0.6779, 0.6645,
         0.8382],
        [0.4836, 0.7808, 0.5472, 0.9814, 0.4623, 0.6954, 0.4916, 0.6250, 0.6790,
         0.8095],
        [0.8847, 0.9108, 0.8454, 0.9396, 0.8425, 0.9406, 0.8233, 0.7667, 0.8254,
         0.6112],
        [0.6781, 0.8835, 0.6773, 0.9707, 0.6486, 0.8234, 0.5419, 0.7178, 0.7217,
         0.7402]])
tensor([0., 1., 1., 1., 1., 1., 0., 1.])


In [ ]:
#now we will create a custom Model for training
# total parameters are fetures * shape of output + bais
HIDDEN_NEURONS = 10
class mymodel(nn.Module):
  def __init__(self):
    super(mymodel,self).__init__()
    self.input_layer= nn.Linear(X.shape[1] , HIDDEN_NEURONS)
    self.linear = nn.Linear (HIDDEN_NEURONS,1)
    self.sigmoid = nn.Sigmoid()

  def forward(self,x):
    x = self.input_layer(x)
    x= self.linear(x)
    x= self.sigmoid(x)
    return x
model = mymodel().to(device)

In [ ]:
summary(mymodel(), (X.shape[1] ,))

----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Linear-1                   [-1, 10]             110
            Linear-2                    [-1, 1]              11
           Sigmoid-3                    [-1, 1]               0
Total params: 121
Trainable params: 121
Non-trainable params: 0
----------------------------------------------------------------
Input size (MB): 0.00
Forward/backward pass size (MB): 0.00
Params size (MB): 0.00
Estimated Total Size (MB): 0.00
----------------------------------------------------------------


In [ ]:
# now we have to do training for that define the loss and the optimizer
losscal = nn.BCELoss()
optimizer  = Adam(model.parameters(), lr=1e-3)
# define the lists we need in the training loop
total_loss_train= []
total_loss_val=[]
total_acc_train = []
total_acc_val = []
epochs= 10

In [ ]:
for epoch in range(epochs):
  train_loss = 0
  train_acc = 0
  val_loss = 0
  val_acc = 0
  for data in train_dataloader:
    input , label = data
    prediction = model(input).squeeze(1)

    batch_loss = losscal(prediction,label)
    train_loss += batch_loss.item()

    acc = (prediction.round()==label).sum().item()
    train_acc+=acc

    batch_loss.backward()
    optimizer.step()
    optimizer.zero_grad()
    break
  break






In [ ]:
from google.colab import _message

notebook = _message.blocking_request("get_ipynb", timeout_sec=10)